# 📊 Student Performance Prediction — Analysis Notebook
**Subjects:** Essentials Of Data Science | Software Engineering | Deep Learning | Technical Training

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'figure.facecolor': '#0D1117',
    'axes.facecolor': '#161B22',
    'axes.edgecolor': '#30363D',
    'axes.labelcolor': '#C9D1D9',
    'xtick.color': '#8B949E',
    'ytick.color': '#8B949E',
    'text.color': '#C9D1D9',
    'grid.color': '#21262D',
    'grid.linewidth': 0.8,
})

ACCENT   = ['#58A6FF','#3FB950','#F78166','#D2A8FF','#FFA657']
GRAD_COLORS = ['#A+','#A','#B+','#B','#C+','#C','#D','#F']

df = pd.read_csv('../data/student_performance.csv')
print(f'Dataset: {len(df)} students | {df["Subject"].nunique()} subjects')
df.head()

## 1. Dataset Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Dataset Overview', fontsize=16, fontweight='bold', color='#E6EDF3', y=1.02)

# Grade distribution
grade_order = ['A+','A','B+','B','C+','C','D','F']
grade_counts = df['Grade'].value_counts().reindex(grade_order).fillna(0)
bar_colors = ['#3FB950','#58A6FF','#D2A8FF','#FFA657','#F78166','#E74C3C','#8B949E','#444C56']
axes[0].bar(grade_counts.index, grade_counts.values, color=bar_colors, edgecolor='none', width=0.7)
axes[0].set_title('Grade Distribution', fontsize=13, pad=12)
axes[0].set_xlabel('Grade'); axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.4)
for i, (g, v) in enumerate(zip(grade_counts.index, grade_counts.values)):
    axes[0].text(i, v+0.3, str(int(v)), ha='center', va='bottom', color='#E6EDF3', fontsize=10)

# Study hours distribution
axes[1].hist(df['Study Hours (per day)'], bins=20, color='#58A6FF', edgecolor='#0D1117', alpha=0.85)
axes[1].set_title('Study Hours Distribution', fontsize=13, pad=12)
axes[1].set_xlabel('Study Hours / Day'); axes[1].set_ylabel('Count')
axes[1].grid(axis='y', alpha=0.4)
axes[1].axvline(df['Study Hours (per day)'].mean(), color='#F78166', linestyle='--', lw=2,
                label=f'Mean: {df["Study Hours (per day)"].mean():.1f}h')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/overview.png', dpi=120, bbox_inches='tight', facecolor='#0D1117')
plt.show()

## 2. Correlation Heatmap

In [ ]:
num_cols = ['Study Hours (per day)','Attendance (%)','Midterm Scaled (0-20)',
            'Best Quiz (0-20)','Lab + Viva (0-10)','Project (0-10)',
            'End-Term Scaled (0-40)','Total Marks (0-100)']
corr = df[num_cols].corr()
short = ['Study\nHours','Attend','Midterm','BestQuiz','Lab\nViva','Project','EndTerm','Total']

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5,
            cbar_kws={'shrink':0.75}, ax=ax,
            xticklabels=short, yticklabels=short,
            annot_kws={'size':9, 'color':'white'})
ax.set_title('Feature Correlation Heatmap', fontsize=14, pad=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/heatmap.png', dpi=120, bbox_inches='tight', facecolor='#0D1117')
plt.show()

## 3. Study Hours vs Performance (per Subject)

In [ ]:
subjects = df['Subject'].unique()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Study Hours vs Total Marks — By Subject', fontsize=15, fontweight='bold', color='#E6EDF3')
axes = axes.flatten()

for i, (subject, ax) in enumerate(zip(subjects, axes)):
    sub = df[df['Subject'] == subject]
    grade_palette = {'A+':'#3FB950','A':'#58A6FF','B+':'#D2A8FF',
                     'B':'#FFA657','C+':'#F78166','C':'#E74C3C','D':'#8B949E','F':'#444C56'}
    for grade, color in grade_palette.items():
        mask = sub['Grade'] == grade
        ax.scatter(sub[mask]['Study Hours (per day)'], sub[mask]['Total Marks (0-100)'],
                   c=color, label=grade, alpha=0.8, s=60, edgecolors='none')

    # Regression line
    x = sub['Study Hours (per day)'].values.reshape(-1,1)
    y = sub['Total Marks (0-100)'].values
    lr = LinearRegression().fit(x, y)
    xr = np.linspace(x.min(), x.max(), 50).reshape(-1,1)
    ax.plot(xr, lr.predict(xr), color='#FF7B72', lw=2, linestyle='--', alpha=0.9, label='Trend')

    r2 = r2_score(y, lr.predict(x))
    ax.set_title(subject, fontsize=11, pad=8)
    ax.set_xlabel('Study Hours/Day'); ax.set_ylabel('Total Marks')
    ax.grid(alpha=0.3); ax.set_ylim(0, 105)
    ax.text(0.97, 0.05, f'R²={r2:.2f}', transform=ax.transAxes,
            ha='right', va='bottom', color='#FF7B72', fontsize=10)

handles = [mpatches.Patch(facecolor=c, label=g)
           for g,c in {'A+':'#3FB950','A':'#58A6FF','B+':'#D2A8FF',
                       'B':'#FFA657','C+':'#F78166','F':'#444C56'}.items()]
fig.legend(handles=handles, loc='lower center', ncol=6, fontsize=9,
           framealpha=0.2, labelcolor='#C9D1D9', facecolor='#161B22', bbox_to_anchor=(0.5, -0.01))
plt.tight_layout()
plt.savefig('../data/study_vs_performance.png', dpi=120, bbox_inches='tight', facecolor='#0D1117')
plt.show()

## 4. Feature Importance

In [ ]:
from sklearn.preprocessing import StandardScaler

feat_cols = ['Study Hours (per day)','Attendance (%)','Midterm Scaled (0-20)',
             'Best Quiz (0-20)','Lab + Viva (0-10)','Project (0-10)']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Feature Importance for End-Term Prediction — By Subject',
             fontsize=14, fontweight='bold', color='#E6EDF3')
axes = axes.flatten()

for i, (subject, ax) in enumerate(zip(subjects, axes)):
    sub = df[df['Subject'] == subject]
    X = sub[feat_cols].fillna(sub[feat_cols].mean())
    y = sub['End-Term Scaled (0-40)']
    sc = StandardScaler()
    Xs = sc.fit_transform(X)
    lr = LinearRegression().fit(Xs, y)
    coefs = pd.Series(lr.coef_, index=feat_cols).sort_values()
    colors = ['#3FB950' if c > 0 else '#F78166' for c in coefs]
    ax.barh(coefs.index, coefs.values, color=colors, edgecolor='none')
    ax.set_title(subject, fontsize=10, pad=8)
    ax.axvline(0, color='#8B949E', lw=0.8, linestyle='-')
    ax.grid(axis='x', alpha=0.3)
    short_labels = ['Study Hrs','Attendance','Midterm','Best Quiz','Lab+Viva','Project']
    ax.set_yticklabels(short_labels, fontsize=8)

plt.tight_layout()
plt.savefig('../data/feature_importance.png', dpi=120, bbox_inches='tight', facecolor='#0D1117')
plt.show()

## 5. Subject-wise Statistics

In [ ]:
stats = df.groupby('Subject').agg(
    Students=('Student Name','count'),
    Avg_Study_Hours=('Study Hours (per day)','mean'),
    Avg_Attendance=('Attendance (%)','mean'),
    Avg_Total=('Total Marks (0-100)','mean'),
    Pass_Rate=('Grade', lambda x: (x!='F').mean()*100)
).round(2)
print(stats.to_string())

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(stats))
w = 0.35
ax.bar(x - w/2, stats['Avg_Total'], w, color='#58A6FF', label='Avg Total Marks', alpha=0.9)
ax.bar(x + w/2, stats['Pass_Rate'], w, color='#3FB950', label='Pass Rate (%)', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels([s.replace(' ', '\n') for s in stats.index], fontsize=9)
ax.set_ylabel('Score / Rate')
ax.set_title('Subject-wise Avg Total Marks vs Pass Rate', fontsize=13, pad=12)
ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 115)
plt.tight_layout()
plt.savefig('../data/subject_stats.png', dpi=120, bbox_inches='tight', facecolor='#0D1117')
plt.show()